In [1]:

# imports
import json
import pandas as pd
from os import path

# constants
intersection_data_path = ("./data/raw/intersections/Jefferson_County_KY_Street_Intersections.geojson")

In [2]:
def get_records(intersection_data_path):
    with open(intersection_data_path, 'r') as file:
        data = json.load(file)
    features = data['features']
    for feature in features:
        properties = feature['properties']
        geometry = feature['geometry']
        # properties['geo_type'] = geometry['type'] # Always == "Point". Not useful.
        properties['GEOMETRY'] = geometry['coordinates']
        yield properties

df =  pd.DataFrame.from_dict(get_records(path.expanduser(intersection_data_path))).convert_dtypes()
intersections_raw = df

bad_row = 100354
# see notes below
# full of nulls that mess up road name compression
# remove row before next step

df = df.drop(df[df.OBJECTID == bad_row].index)

df.head()

,OBJECTID,SIFCODE1,SIFCODE2,INTID,SCCAD_ID,FST_INTPRE,FST_INTNAME,FST_INTSUF,SEC_INTPRE,SEC_INTNAME,SEC_INTSUF,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GLOBALID,GEOMETRY
0,1,5464,7662,154647662,1,,REHL,RD,W,REHL,CT,1278243.0,259531.9375,4976,6856,{CD0D9D51-41FB-46FF-B229-AC9C0DDB7E61},"[-85.51044384084946, 38.20588660809318]"
1,2,5464,6551,254646551,2,,REHL,RD,,TUCKER STATION,RD,1273126.375,257588.25,4976,5908,{A9FAED81-F7AE-436F-A657-F95FE1B905E8},"[-85.52816882852979, 38.20037561255241]"
2,3,5464,6551,354646551,3,,REHL,RD,,TUCKER STATION,RD,1273050.50875,257590.85375,4976,5908,{71155AF3-3487-4EDA-8B72-B282378DE7F3},"[-85.52841872335158, 38.200360349057064]"
3,4,3194,9996,431949996,4,,I 64 EAST,,,I 265 RAMP,,1279903.25,265597.5,3076,8763,{AD332CAB-27B0-48B7-ACBF-5CDEEC31654B},"[-85.50495414554669, 38.22260344055154]"
4,5,9349,9996,593499996,5,,I 265 NORTH,,,I 265 RAMP,,1279730.75,265426.4375,8197,8763,{0FE38BAB-8B39-4BE8-BA57-1DAA46FDD4BC},"[-85.50554642815675, 38.222127288727606]"


In [3]:
## find a good index for data

## SCCAD_ID -> No
df.SCCAD_ID.is_unique # False
vc = df.SCCAD_ID.value_counts()
sc2 = vc[vc > 1].index # SCCAD_ID s that point to more than one intersection

df[df.SCCAD_ID.isin(sc2)]


## OBJECTID
df.OBJECTID.is_unique # True


## GLOBALID

  # annoying to look at and use:
  # strings like this: {CD0D9D51-41FB-46FF-B229-AC9C0DDB7E61}

df.GLOBALID.is_unique # True
df.GLOBALID.apply(hash) # negative numbers


## INTID ->  best choice

 # derived from SCCAD_ID + SIFCODES 1 and 2

df.INTID.is_unique # True: can use as index?

df.INTID.str.strip().is_unique # still true

df.INTID.apply(hash) # includes negative numbers
set(''.join(df.INTID.str.strip())) # ->
 # {'0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'C', 'D', 'E', 'F'}
 # only hex chars

 # INTID is a string but it cen be expressed as a number.

df.INTID.apply(lambda x:int(x, base=16)).is_unique # True

"""Conclusion: 
  drop OBJECTID, GLOBALID, SCCAD_ID
  convert INTID to integer and use the result as index"""


def set_index(df:pd.DataFrame) -> pd.DataFrame:
    # drop unneeded id columns
    df = df.drop(['OBJECTID', 'SCCAD_ID', 'GLOBALID'], axis=1)

    # convert INTID string to integers.
    # TODO / NOTE is this necessary? Does the other code expect the index to be an integer?
    df.INTID = df.INTID.apply(lambda x:int(x, base=16))

    # set the index and return dataframe
    return df.set_index("INTID")

df1 = set_index(df)
df1.head()

,SIFCODE1,SIFCODE2,FST_INTPRE,FST_INTNAME,FST_INTSUF,SEC_INTPRE,SEC_INTNAME,SEC_INTSUF,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GEOMETRY
INTID,,,,,,,,,,,,,
5710837346,5464,7662,,REHL,RD,W,REHL,CT,1278243.0,259531.9375,4976,6856,"[-85.51044384084946, 38.20588660809318]"
10005800273,5464,6551,,REHL,RD,,TUCKER STATION,RD,1273126.375,257588.25,4976,5908,"[-85.52816882852979, 38.20037561255241]"
14300767569,5464,6551,,REHL,RD,,TUCKER STATION,RD,1273050.50875,257590.85375,4976,5908,"[-85.52841872335158, 38.200360349057064]"
18011691414,3194,9996,,I 64 EAST,,,I 265 RAMP,,1279903.25,265597.5,3076,8763,"[-85.50495414554669, 38.22260344055154]"
23945910678,9349,9996,,I 265 NORTH,,,I 265 RAMP,,1279730.75,265426.4375,8197,8763,"[-85.50554642815675, 38.222127288727606]"


In [4]:
# Compress road name info

def compress_roadnames(df:pd.DataFrame) -> pd.DataFrame:
    fst_road_info = df[["FST_INTPRE", "FST_INTNAME", "FST_INTSUF"]]
    df['FST_ROADNAME'] = fst_road_info.apply(" ".join, axis=1).str.strip()
    
    sec_road_info = df[["SEC_INTPRE", "SEC_INTNAME", "SEC_INTSUF"]]
    df['SEC_ROADNAME'] = sec_road_info.apply(" ".join, axis=1).str.strip()

    return df.drop(["FST_INTPRE", "FST_INTNAME", "FST_INTSUF", 
                  "SEC_INTPRE", "SEC_INTNAME", "SEC_INTSUF"], axis=1)
    
df2 = compress_roadnames(df1)
df2.head()


,SIFCODE1,SIFCODE2,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GEOMETRY,FST_ROADNAME,SEC_ROADNAME
INTID,,,,,,,,,
5710837346,5464,7662,1278243.0,259531.9375,4976,6856,"[-85.51044384084946, 38.20588660809318]",REHL RD,W REHL CT
10005800273,5464,6551,1273126.375,257588.25,4976,5908,"[-85.52816882852979, 38.20037561255241]",REHL RD,TUCKER STATION RD
14300767569,5464,6551,1273050.50875,257590.85375,4976,5908,"[-85.52841872335158, 38.200360349057064]",REHL RD,TUCKER STATION RD
18011691414,3194,9996,1279903.25,265597.5,3076,8763,"[-85.50495414554669, 38.22260344055154]",I 64 EAST,I 265 RAMP
23945910678,9349,9996,1279730.75,265426.4375,8197,8763,"[-85.50554642815675, 38.222127288727606]",I 265 NORTH,I 265 RAMP


Why are FST_SIFID and SEC_SIFID floats on import?

-> because of a NAN value in item where OBJECTID == 100354
bad_row = 100354

both CSV and JSON are like this

#### CSV:
```csv
X,Y,OBJECTID,SIFCODE1,SIFCODE2,INTID,SCCAD_ID,FST_INTPRE,FST_INTNAME,FST_INTSUF,SEC_INTPRE,SEC_INTNAME,SEC_INTSUF,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GLOBALID
```

1227948.0,272835.375,100354,,,,,,,,,,,1227948,272835.375,,,{60D18EC3-BBBC-419B-8A10-62C37F39E987}

#### JSON:
```json
null = float('nan')
{ "type": "Feature",
  "properties": {
     "OBJECTID": 100354, "SIFCODE1": null, "SIFCODE2": null, "INTID": null, "SCCAD_ID": null,
     "FST_INTPRE": null, "FST_INTNAME": null, "FST_INTSUF": null, "SEC_INTPRE": null, "SEC_INTNAME": null,
     "SEC_INTSUF": null, "X_COORD": 1227948.0, "Y_COORD": 272835.375, "FST_SIFID": null, "SEC_SIFID": null,
     "GLOBALID": "{60D18EC3-BBBC-419B-8A10-62C37F39E987}" },
      
  "geometry": { "type": "Point", "coordinates": [ -85.686176099576869, 38.240392433657043 ] } }
```

This interferes with some code that simplifies the road names. Remove the row with nulls and any other before further processing. 


In [5]:
# convert coordinates from LOJIC CRS to (longitude, latitude)
# LOJIC projection: ESRI:102679
# NAD_1983_StatePlane_Kentucky_North_FIPS_1601_Feet

# Standard long, lat: epsg:4326

import numpy as np

from pyproj import CRS
from pyproj.transformer import Transformer

def convert_state_grid_coordinates(df):
    # get projection info, create transformer
    KY_grid_CRS = CRS("ESRI:102679")
    long_lat_CRS = CRS("epsg:4326")
    CRS_transformer = Transformer.from_crs(crs_from=KY_grid_CRS, crs_to=long_lat_CRS, always_xy=True)

    # apply transformer to X, Y coordinates; add result to dataframe
    conversion = df['X_COORD'].combine(df['Y_COORD'], CRS_transformer.transform)
    return conversion

    # drop now unneeded X, Y coordinate columns and return dataframe
    return df.drop(["X_COORD", "Y_COORD"], axis=1)

state_grid_conversion = convert_state_grid_coordinates(df2)
state_grid_conversion

df3 = df2.drop(["X_COORD", "Y_COORD"], axis=1)
df3['state_grid_GEO'] = state_grid_conversion
#df3
    
  

In [6]:
# # county box
# west_longitude = -85.94712712079293
# east_longitude = -85.3443621648922
# south_latitude = 37.99712528351634
# north_latitude = 38.38023822809115

# delta_long = abs(east_longitude - west_longitude)
# delta_lat = abs(north_latitude - south_latitude)

# # # old point distance code that worked pretyt well


# long_dist = 30
# lat_dist = 26

# def n(point):
#     long, lat = point
#     v = ((long - west_longitude) / delta_long), ((lat - south_latitude) / delta_lat)
#     return np.array(v)

# def d(point):
#     long, lat = point
#     long = (long*long_dist)**2
#     lat = (lat*lat_dist)**2
#     return np.sqrt(long + lat)

# oft = (long_lat_coordinates.apply(n) - intersections.GEOMETRY.apply(n)).apply(d)*5480

# def dd(point):
#     long, lat = point
#     return (long/delta_long), (lat/delta_lat)

# ft = (long_lat_coordinates - intersections.GEOMETRY).apply(dd).apply(d)*5280

# ft.describe()
# ft[ft>=10]

# oft[oft>=100]

# ... stage before reorganizing column names

KY_grid_CRS = CRS("ESRI:102679")
long_lat_CRS = CRS("epsg:4326")

long_lat_to_grid = Transformer.from_crs(crs_from=long_lat_CRS, crs_to=KY_grid_CRS, always_xy=True)
ll2g_T = long_lat_to_grid.transform



# we have...
state_grid_conversion = state_grid_conversion.apply(np.array)
coordinates = df3.GEOMETRY.apply(np.array)

cc = (coordinates - state_grid_conversion)
cc.apply(np.linalg.norm).sort_values(ascending=True)

from common.county_geometry import delta_longitude, longitude_span_mi, delta_latitude, latitude_span_mi
from common.county_geometry import haversine_distance_mi as hav
from common.county_geometry import west_longitude as min_longitude
from common.county_geometry import south_latitude as min_latitude
from common.county_geometry import central_angle, earth_radius_mi, earth_radius_km, km_to_miles
from operator import itemgetter


pd.concat((
coordinates.combine(state_grid_conversion, hav),
cc.apply(np.linalg.norm)), axis=1)

def g(point):
    return hav((0,0), point)

pd.concat((
coordinates.combine(state_grid_conversion, hav),
cc.apply(np.linalg.norm),
cc.apply(g)), axis=1)


,0,1,2
INTID,,,
5710837346,0.000567,0.000009,0.000603
10005800273,0.002120,0.000033,0.002270
14300767569,0.000567,0.000009,0.000603
18011691414,0.000567,0.000009,0.000603
23945910678,0.000567,0.000009,0.000603
...,...,...,...
847259462915456,0.000568,0.000009,0.000604
847265632743817,0.000567,0.000009,0.000603
847261337735316,0.000567,0.000009,0.000603


##### Info about X_COORD YCOORD system

via: https://www.lojic.org/data/projection-information

For this system, the Commonwealth shall be divided into a north zone and a south zone. The north zone shall be a Lambert conformal conic projection of the North American Datum of 1983, having standard parallels at north latitudes 37 degrees, 58 minutes, and 38 degrees, 58 minutes along which parallels the scale shall be exact. The origin of coordinates shall be at the intersection of the meridian 84 degrees, 15 minutes west of Greenwich, and the parallel 37 degrees, 30 minutes north latitude. This origin shall be given the coordinates: N=0, E=500,000.000 meters. The south zone shall be a Lambert conformal conic projection of the North American Datum of 1983, having standard parallels at north latitudes 36 degrees, 44 minutes, and 37 degrees, 56 minutes along which parallels the scale shall be exact. The origin of coordinates shall be at the intersection of the meridian 85 degrees, 45 minutes west of Greenwich, and the parallel 36 degrees, 20 minutes north latitude. This origin shall be given the coordinates: N=500,000.000, E=500,000.000 meters. The southern edge of the following counties shall delineate the boundary between the north zone and the south zone: Bullitt, Spencer, Anderson, Woodford, Jessamine, Fayette, Clark, Montgomery, Menifee, Morgan, and Lawrence.

One U. S. survey foot equals (1200)/(3937) meter. For conversion of meters to U. S. survey feet, multiply the meters by 3.28083333333 to twelve (12) significant figures. When converting from meters to feet, the conversion factor defined by the U. S. survey foot shall be used.


The plane coordinate values for a point on the earth's surface, used to express the geographic position or location of the point in the appropriate zone of this system, shall consist of two (2) distances expressed in U. S. survey feet and decimals of a foot when using the Kentucky Coordinate System of 1983. For the Kentucky Coordinate System of 1983, one (1) of the distances, to be known as the "northing" or "N", shall give the position in a north/south direction. The other, to be known as the "easting" or "E" shall give the position in an east/west direction. These coordinates shall be made to depend upon and conform to plane rectangular coordinates values for the monumented points of the North American National Geodetic Horizontal Network as published by the National Ocean Service/National Geodetic Survey, and whose plane coordinates have been computed on the systems established by the National Ocean Service/National Geodetic Survey. Any such station may be used for establishing a survey connection to the Kentucky Coordinate System of 1983.

In [7]:
xy = df2[['X_COORD', 'Y_COORD']]

def convert_long_lat_to_grid(point):
    return ll2g_T(*point)

llconv = coordinates.apply(convert_long_lat_to_grid)
llconv = llconv.transform({"X":itemgetter(0), "Y":itemgetter(1)})

gf = pd.concat((xy, llconv), axis=1)
gf['x_diff'] = x_diff = (xy.X_COORD - llconv.X)
gf['y_diff'] = y_diff = (xy.Y_COORD - llconv.Y)


d = np.sqrt(x_diff**2 + y_diff**2)

d

INTID
5710837346          2.994226
10005800273        11.190484
14300767569         2.994305
18011691414         2.995155
23945910678         2.995139
                     ...    
847259462915456     2.998237
847265632743817     2.994492
847261337735316     2.994492
847274257577475     2.992074
847272347859222      2.99181
Length: 20946, dtype: Float64

In [25]:

from common.county_geometry import earth_radius_km_2

erm = earth_radius_mi
dd = (state_grid_conversion.combine(coordinates, central_angle)*erm*5280) - d

dd[dd > .01]

INTID
7109094830676      0.011145
26471278204726     0.134185
53219299825282     0.015549
56153365573778     0.018789
92882165092470     0.070796
                     ...   
721429722493524    0.011828
721498442069316    0.031087
722688151655027    0.015558
728980310464018    0.013095
730311764119607    0.046618
Length: 124, dtype: Float64

In [28]:
from common import county_geometry

hav = county_geometry.haversine_distance_mi

dd = df3.GEOMETRY.combine(df3.state_grid_GEO, hav)
dd


INTID
5710837346         0.000567
10005800273        0.002120
14300767569        0.000567
18011691414        0.000567
23945910678        0.000567
                     ...   
847259462915456    0.000568
847265632743817    0.000567
847261337735316    0.000567
847274257577475    0.000567
847272347859222    0.000567
Length: 20946, dtype: float64

In [29]:
def clean_index(df:pd.DataFrame) -> None:
    # convert INTID string to integers.
    # TODO / NOTE is this necessary? Does the other code expect the index to be an integer?
    df.INTID = df.INTID.apply(lambda x:int(x, base=16))

    # drop unneeded id columns
    return df.drop(['OBJECTID', 'SCCAD_ID', 'GLOBALID'], axis=1)


def create_clean_df(df):
    keep_columns = ['INTID', 'FST_ROADNAME', 'SEC_ROADNAME', 'SIFCODE1', 'SIFCODE2', 
    #keep_columns = ['FST_ROADNAME', 'SEC_ROADNAME', 'SIFCODE1', 'SIFCODE2', 
                'FST_SIFID', 'SEC_SIFID', 'GEOMETRY', 'state_grid_GEO']
    
    bad_row = 100354
    # full of nulls that would mess up subsequent steps
    df = df.drop(df[df.OBJECTID == bad_row].index)
    
    df = clean_index(df)
    #df = df.set_index("INTID")
    df = compress_roadnames(df)
    df = convert_state_grid_coordinates(df)
    return df[keep_columns].convert_dtypes()

intersections_clean = create_clean_df(intersections_raw)
display(intersections_clean.head())



,INTID,FST_ROADNAME,SEC_ROADNAME,SIFCODE1,SIFCODE2,FST_SIFID,SEC_SIFID,GEOMETRY,state_grid_GEO
0,5710837346,REHL RD,W REHL CT,5464,7662,4976,6856,"[-85.51044384084946, 38.20588660809318]","(-85.51043903006253, 38.205879314617064)"
1,10005800273,REHL RD,TUCKER STATION RD,5464,6551,4976,5908,"[-85.52816882852979, 38.20037561255241]","(-85.52814980553048, 38.20034879942594)"
2,14300767569,REHL RD,TUCKER STATION RD,5464,6551,4976,5908,"[-85.52841872335158, 38.200360349057064]","(-85.52841390771789, 38.20035305747533)"
3,18011691414,I 64 EAST,I 265 RAMP,3194,9996,3076,8763,"[-85.50495414554669, 38.22260344055154]","(-85.5049493351228, 38.22259614360763)"
4,23945910678,I 265 NORTH,I 265 RAMP,9349,9996,8197,8763,"[-85.50554642815675, 38.222127288727606]","(-85.50554161759499, 38.22211999190267)"


In [32]:

# write transformed data to file.
store_path = "./data/cleaner/intersections_clean.json"

intersections_clean.to_json(store_path, orient='records', lines=True, double_precision=15)


In [31]:
def read_in_intersections(path_to_intersection_data):
    df = pd.read_json(path_to_intersection_data, orient='records', lines=True)
    df['GEOMETRY'] = df.GEOMETRY.apply(np.array)
    df['state_grid_GEO'] = df.state_grid_GEO.apply(np.array)
    return df.set_index("INTID")

data = read_in_intersections(store_path)
data.dtypes
data.head()


,FST_ROADNAME,SEC_ROADNAME,SIFCODE1,SIFCODE2,FST_SIFID,SEC_SIFID,GEOMETRY,state_grid_GEO
INTID,,,,,,,,
5710837346,REHL RD,W REHL CT,5464,7662,4976,6856,"[-85.51044384084946, 38.20588660809318]","[-85.51043903006253, 38.205879314617064]"
10005800273,REHL RD,TUCKER STATION RD,5464,6551,4976,5908,"[-85.52816882852979, 38.20037561255241]","[-85.52814980553048, 38.20034879942594]"
14300767569,REHL RD,TUCKER STATION RD,5464,6551,4976,5908,"[-85.52841872335158, 38.200360349057064]","[-85.52841390771789, 38.20035305747533]"
18011691414,I 64 EAST,I 265 RAMP,3194,9996,3076,8763,"[-85.50495414554669, 38.22260344055154]","[-85.5049493351228, 38.22259614360763]"
23945910678,I 265 NORTH,I 265 RAMP,9349,9996,8197,8763,"[-85.50554642815675, 38.222127288727606]","[-85.50554161759499, 38.22211999190267]"
